
**Generate_sample_dataset.py**
==========================


Generates a synthetic sample of the Credit Card Fraud detection and analysis dataset
(same schema as the Kaggle dataset) for GitHub upload.


WHY THIS EXISTS:

    **The original creditcard.csv is ~150MB (284,807 rows)** — too large for GitHub.
    This script creates a 10,000-row synthetic sample with identical column
    structure and realistic statistical properties, safe to commit.

USAGE:

     python generate_sample_dataset.py

OUTPUT:

     creditcard_sample.csv  (~1.5 MB, GitHub-friendly)

NOTE:

  For real model training, download the full dataset from:
  https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
"""

In [ ]:


import pandas as pd
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
N_NORMAL      = 9700     # normal transactions
N_FRAUD       = 300      # fraud transactions (~3% rate, vs 0.17% real)
RANDOM_SEED   = 42
OUTPUT_FILE   = "creditcard_sample.csv"

# ── Helpers ───────────────────────────────────────────────────────────────────
def make_v_features(n: int, is_fraud: bool) -> dict:
    """
    Generate V1–V28 PCA-like features.
    Key fraud features are shifted based on real-dataset correlations
    (V14, V12, V10, V17 are the strongest fraud signals).
    """
    # Mean shifts for fraud class (from real dataset analysis)
    fraud_shifts = {
        "V1":  -3.5, "V2":  2.5, "V3":  -4.0, "V4":   2.5,
        "V7":  -3.0, "V9":  -2.5, "V10": -4.0, "V11":  2.5,
        "V12": -4.5, "V14": -5.0, "V16": -2.5, "V17": -4.0,
        "V18": -2.5, "V19":  1.5,
    }
    data = {}
    for i in range(1, 29):
        key   = f"V{i}"
        shift = fraud_shifts.get(key, 0.0) if is_fraud else 0.0
        data[key] = np.random.normal(loc=shift, scale=1.2, size=n)
    return data


def generate_sample(n_normal: int, n_fraud: int, seed: int) -> pd.DataFrame:
    np.random.seed(seed)

    # Normal transactions
    normal_data = make_v_features(n_normal, is_fraud=False)
    normal_df   = pd.DataFrame(normal_data)
    normal_df["Time"]   = np.sort(np.random.uniform(0, 172_792, n_normal))
    normal_df["Amount"] = np.abs(np.random.lognormal(3.5, 1.5, n_normal)).clip(0.01, 5000)
    normal_df["Class"]  = 0

    # Fraud transactions
    fraud_data = make_v_features(n_fraud, is_fraud=True)
    fraud_df   = pd.DataFrame(fraud_data)
    fraud_df["Time"]   = np.random.uniform(0, 172_792, n_fraud)
    fraud_df["Amount"] = np.abs(np.random.lognormal(2.5, 1.8, n_fraud)).clip(0.01, 2500)
    fraud_df["Class"]  = 1

    # Combine & shuffle
    df = (
        pd.concat([normal_df, fraud_df], ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    # Match original column order: Time, V1..V28, Amount, Class
    v_cols    = [f"V{i}" for i in range(1, 29)]
    df        = df[["Time"] + v_cols + ["Amount", "Class"]]

    # Round to match original dataset precision
    df[v_cols]    = df[v_cols].round(6)
    df["Amount"]  = df["Amount"].round(2)
    df["Time"]    = df["Time"].round(0).astype(int)
    df["Class"]   = df["Class"].astype(int)

    return df


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("Generating synthetic credit card fraud sample dataset...")
    print(f"  Normal transactions : {N_NORMAL:,}")
    print(f"  Fraud transactions  : {N_FRAUD:,}")
    print(f"  Total rows          : {N_NORMAL + N_FRAUD:,}")

    df = generate_sample(N_NORMAL, N_FRAUD, RANDOM_SEED)

    # ── Validation ────────────────────────────────────────────────────────────
    assert df.shape == (N_NORMAL + N_FRAUD, 31),   "Shape mismatch!"
    assert df.isnull().sum().sum() == 0,            "Unexpected nulls!"
    assert set(df["Class"].unique()) == {0, 1},     "Class values wrong!"
    assert list(df.columns[:3]) == ["Time","V1","V2"], "Column order wrong!"

    # ── Save ──────────────────────────────────────────────────────────────────
    df.to_csv(OUTPUT_FILE, index=False)

    file_size_kb = __import__("os").path.getsize(OUTPUT_FILE) / 1024
    fraud_pct    = df["Class"].mean() * 100

    print(f"\n✅ Saved: {OUTPUT_FILE}")
    print(f"   Rows        : {len(df):,}")
    print(f"   Columns     : {df.shape[1]}")
    print(f"   Fraud rate  : {fraud_pct:.2f}%")
    print(f"   File size   : {file_size_kb:.1f} KB  (GitHub limit: 100 MB)")
    print(f"\n  Amount stats:")
    print(f"    Normal — mean: €{df[df['Class']==0]['Amount'].mean():.2f}  "
          f"max: €{df[df['Class']==0]['Amount'].max():.2f}")
    print(f"    Fraud  — mean: €{df[df['Class']==1]['Amount'].mean():.2f}  "
          f"max: €{df[df['Class']==1]['Amount'].max():.2f}")
    print(f"\n  Preview:")
    print(df[["Time","V1","V14","Amount","Class"]].head(5).to_string(index=False))

Generating synthetic credit card fraud sample dataset...
  Normal transactions : 9,700
  Fraud transactions  : 300
  Total rows          : 10,000

✅ Saved: creditcard_sample.csv
   Rows        : 10,000
   Columns     : 31
   Fraud rate  : 3.00%
   File size   : 2708.0 KB  (GitHub limit: 100 MB)

  Amount stats:
    Normal — mean: €102.12  max: €5000.00
    Fraud  — mean: €55.23  max: €1752.03

  Preview:
  Time        V1       V14  Amount  Class
111545  3.165458  0.916824   47.94      0
 83870 -1.244479 -2.076247    4.87      0
 30899 -0.419180 -0.974421 1064.94      0
 84790  0.001943 -2.263788  121.24      0
 80691  1.289200 -0.377612   17.61      0
